# Normalización completa del dataset de clima de Galicia

## 1. Importar librerías necesarias

In [4]:
import pandas as pd
import numpy as np
import unicodedata
from pathlib import Path
from difflib import get_close_matches

## 2. Cargar el dataset de clima a normalizar

In [5]:
# Definir la ruta del dataset de clima
dataset_path = r'C:\00 - Proyecto Incendios Galicia - END\data\02 - Municipio normalizado\03 - meteorologia\01 - clima municipios normalizados.csv'

# Cargar el archivo CSV
df = pd.read_csv(dataset_path)

print(f'Filas cargadas: {len(df)}')
print('Columnas disponibles:', list(df.columns))
print(f'Tamaño del dataset: {df.shape[0]} filas x {df.shape[1]} columnas')
df.head()

C:\Users\Jacinto\AppData\Local\Temp\ipykernel_2800\2247843127.py:5: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(dataset_path)


Filas cargadas: 2631138
Columnas disponibles: ['time', 'tavg', 'tmin', 'tmax', 'prcp', 'municipio', 'proveniente_de']
Tamaño del dataset: 2631138 filas x 7 columnas


,time,tavg,tmin,tmax,prcp,municipio,proveniente_de
0,2000-01-01,NaN,-5.8,9.6,0.0,abadín,NaN
1,2000-01-02,NaN,-4.4,6.2,0.0,abadín,NaN
2,2000-01-03,NaN,-4.4,11.2,0.0,abadín,NaN
3,2000-01-04,NaN,5.8,9.6,0.3,abadín,NaN
4,2000-01-05,NaN,3.2,11.2,0.0,abadín,NaN


In [6]:
# Renombrar columnas a nombres claros y descriptivos
columnas_renombrar = {
    'time': 'fecha',
    'tavg': 'temp_media',
    'tmin': 'temp_min',
    'tmax': 'temp_max',
    'prcp': 'precipitacion',
    'snow': 'nieve',
    'wdir': 'direccion_viento',
    'wspd': 'velocidad_viento',
    'wpgt': 'rafaga_viento',
    'pres': 'presion',
    'tsun': 'horas_sol',
    'municipio': 'municipio',
    'archivo_origen': 'archivo_origen',
    'proveniente_de': 'fuente_datos',
    'municipio_origen_datos': 'municipio_fuente',
    'distancia_origen_km': 'distancia_fuente_km',
}

print('Nombres originales de columnas:')
print(list(df.columns))

df.rename(columns=columnas_renombrar, inplace=True)
df.columns = [col.lower() for col in df.columns]

print('Nombres de columnas tras la normalización:')
print(list(df.columns))

Nombres originales de columnas:
['time', 'tavg', 'tmin', 'tmax', 'prcp', 'municipio', 'proveniente_de']
Nombres de columnas tras la normalización:
['fecha', 'temp_media', 'temp_min', 'temp_max', 'precipitacion', 'municipio', 'fuente_datos']


In [7]:
# Normalizar la columna 'fecha' a tipo datetime y dejar solo la fecha (sin hora)
if 'fecha' in df.columns:
    df['fecha'] = pd.to_datetime(df['fecha'], errors='coerce').dt.date
    print("Columna 'fecha' normalizada. Ejemplo de valores:")
    print(df['fecha'].dropna().astype(str).unique()[:5])

Columna 'fecha' normalizada. Ejemplo de valores:
['2000-01-01' '2000-01-02' '2000-01-03' '2000-01-04' '2000-01-05']


In [8]:
# Filtrar registros solo entre el 1 de enero de 2000 y el 31 de diciembre de 2022
fecha_inicio = pd.to_datetime('2000-01-01').date()
fecha_fin = pd.to_datetime('2022-12-31').date()
df = df[(df['fecha'] >= fecha_inicio) & (df['fecha'] <= fecha_fin)]
print(f'Registros tras filtrar por fecha: {len(df)}')

Registros tras filtrar por fecha: 2631138


In [9]:
# Mostrar la cantidad de valores nulos por columna
print('Valores nulos por columna:')
print(df.isnull().sum())

Valores nulos por columna:
fecha                  0
temp_media        604252
temp_min           14721
temp_max           14729
precipitacion      32548
municipio              0
fuente_datos     1782637
dtype: int64


In [10]:
# Guardar el dataset limpio en la ruta indicada
import os
ruta_export = r'C:\00 - Proyecto Incendios Galicia - END\data\03 - Normalizado completo\03 - meteorologia'
os.makedirs(ruta_export, exist_ok=True)
archivo_export = os.path.join(ruta_export, '01 - clima normalizado completo.csv')
df.to_csv(archivo_export, index=False, encoding='utf-8')
print(f'Dataset limpio guardado en: {archivo_export}')

Dataset limpio guardado en: C:\00 - Proyecto Incendios Galicia - END\data\03 - Normalizado completo\03 - meteorologia\01 - clima normalizado completo.csv


### Normalización de tipos tras la exportación del dataset de clima
Ejecuta la siguiente celda para revisar y normalizar los tipos de datos del archivo final exportado.

In [11]:
# Cargar y normalizar tipos del dataset final de clima
import pandas as pd

archivo_export = r'C:\00 - Proyecto Incendios Galicia - END\data\03 - Normalizado completo\03 - meteorologia\01 - clima normalizado completo.csv'

# Cargar con low_memory=False para evitar warnings
df_check = pd.read_csv(archivo_export, low_memory=False)

# Convertir fecha
if 'fecha' in df_check.columns:
    df_check['fecha'] = pd.to_datetime(df_check['fecha'], errors='coerce').dt.date

# Convertir a numérico las columnas que deberían serlo (ajusta la lista según tus necesidades)
cols_numericas = [
    'temp_media', 'temp_min', 'temp_max', 'precipitacion', 'nieve',
    'direccion_viento', 'velocidad_viento', 'rafaga_viento', 'presion', 'horas_sol',
    'distancia_fuente_km'
 ]
for col in cols_numericas:
    if col in df_check.columns:
        df_check[col] = pd.to_numeric(df_check[col].astype(str).str.replace(',', '.', regex=False), errors='coerce')

# Revisar tipos finales
print('Tipos de datos tras normalización:')
print(df_check.dtypes)
df_check.head()

Tipos de datos tras normalización:
fecha             object
temp_media       float64
temp_min         float64
temp_max         float64
precipitacion    float64
municipio         object
fuente_datos      object
dtype: object


,fecha,temp_media,temp_min,temp_max,precipitacion,municipio,fuente_datos
0,2000-01-01,NaN,-5.8,9.6,0.0,abadín,NaN
1,2000-01-02,NaN,-4.4,6.2,0.0,abadín,NaN
2,2000-01-03,NaN,-4.4,11.2,0.0,abadín,NaN
3,2000-01-04,NaN,5.8,9.6,0.3,abadín,NaN
4,2000-01-05,NaN,3.2,11.2,0.0,abadín,NaN


In [12]:
# Guardar el DataFrame normalizado de clima con tipos corregidos en un nuevo archivo
ruta_export_final = r'C:\00 - Proyecto Incendios Galicia - END\data\03 - Normalizado completo\03 - meteorologia'
archivo_export_final = ruta_export_final + '\\02 - clima normalizado completo final.csv'
df_check.to_csv(archivo_export_final, index=False, encoding='utf-8')
print(f'Dataset final normalizado guardado en: {archivo_export_final}')

Dataset final normalizado guardado en: C:\00 - Proyecto Incendios Galicia - END\data\03 - Normalizado completo\03 - meteorologia\02 - clima normalizado completo final.csv


In [ ]:
# Buscar el municipio "A Gudiña" en el dataset guardado
municipio_buscar = "a gudiña"

print(f'Buscando municipio: "{municipio_buscar}"')
print('=' * 50)

# Buscar coincidencias exactas (case insensitive)
coincidencias_exactas = df_check[df_check['municipio'].str.lower() == municipio_buscar.lower()]
print(f'Coincidencias exactas: {len(coincidencias_exactas)}')

# Buscar coincidencias parciales (que contengan el texto)
coincidencias_parciales = df_check[df_check['municipio'].str.lower().str.contains(municipio_buscar.lower(), na=False)]
print(f'Coincidencias parciales: {len(coincidencias_parciales)}')

# Mostrar municipios únicos que contienen "gudiña"
municipios_gudiña = df_check[df_check['municipio'].str.lower().str.contains('gudiña', na=False)]['municipio'].unique()
print(f'Municipios que contienen "gudiña": {list(municipios_gudiña)}')

# Mostrar algunos registros si hay coincidencias
if len(coincidencias_exactas) > 0:
    print('\nPrimeros 5 registros de coincidencias exactas:')
    print(coincidencias_exactas[['municipio', 'fecha', 'temp_media', 'precipitacion']].head())
    datos_gudiña = coincidencias_exactas
elif len(coincidencias_parciales) > 0:
    print('\nPrimeros 5 registros de coincidencias parciales:')
    print(coincidencias_parciales[['municipio', 'fecha', 'temp_media', 'precipitacion']].head())
    datos_gudiña = coincidencias_parciales
else:
    print('\nNo se encontraron coincidencias')
    print('Mostrando algunos municipios para referencia:')
    print(list(df_check['municipio'].unique()[:10]))
    datos_gudiña = None

# EXPORTAR DATOS DE A GUDIÑA A ARCHIVO TXT
if datos_gudiña is not None and len(datos_gudiña) > 0:
    import os
    from datetime import datetime
    
    # Crear directorio de exportación si no existe
    ruta_exportacion = r'C:\00 - Proyecto Incendios Galicia - END\data\reportes'
    os.makedirs(ruta_exportacion, exist_ok=True)
    
    # Nombre del archivo con timestamp
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    archivo_txt = os.path.join(ruta_exportacion, f'datos_a_gudiña_clima_{timestamp}.txt')
    
    # Seleccionar columnas relevantes para el reporte
    columnas_reporte = ['municipio', 'fecha', 'temp_media', 'temp_min', 'temp_max', 
                       'precipitacion', 'velocidad_viento', 'presion', 'horas_sol']
    
    # Filtrar solo las columnas que existen en el dataset
    columnas_disponibles = [col for col in columnas_reporte if col in datos_gudiña.columns]
    datos_reporte = datos_gudiña[columnas_disponibles].copy()
    
    # Ordenar por fecha
    if 'fecha' in datos_reporte.columns:
        datos_reporte = datos_reporte.sort_values('fecha')
    
    # Escribir archivo TXT con formato tabla
    with open(archivo_txt, 'w', encoding='utf-8') as f:
        f.write("=" * 100 + "\n")
        f.write("DATOS CLIMATOLÓGICOS DE A GUDIÑA\n")
        f.write("=" * 100 + "\n")
        f.write(f"Fecha de exportación: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
        f.write(f"Total de registros: {len(datos_reporte):,}\n")
        f.write(f"Municipio encontrado: {municipios_gudiña[0] if len(municipios_gudiña) > 0 else 'A Gudiña'}\n")
        
        # Calcular porcentaje de datos disponibles por columna
        f.write("\nPORCENTAJE DE DATOS DISPONIBLES POR VARIABLE:\n")
        f.write("-" * 60 + "\n")
        total_registros = len(datos_reporte)
        for col in columnas_disponibles:
            if col != 'municipio':  # Excluir municipio del análisis de completitud
                datos_no_nulos = datos_reporte[col].notna().sum()
                porcentaje = (datos_no_nulos / total_registros * 100) if total_registros > 0 else 0
                f.write(f"  • {col:<20}: {datos_no_nulos:>6,} / {total_registros:,} ({porcentaje:>6.2f}%)\n")
        
        # Calcular porcentaje general de completitud
        total_celdas = len(datos_reporte) * len([col for col in columnas_disponibles if col != 'municipio'])
        celdas_con_datos = 0
        for col in columnas_disponibles:
            if col != 'municipio':
                celdas_con_datos += datos_reporte[col].notna().sum()
        
        porcentaje_general = (celdas_con_datos / total_celdas * 100) if total_celdas > 0 else 0
        f.write("-" * 60 + "\n")
        f.write(f"COMPLETITUD GENERAL: {celdas_con_datos:,} / {total_celdas:,} ({porcentaje_general:.2f}%)\n")
        
        f.write("=" * 100 + "\n\n")
        
        # Escribir encabezados
        f.write("DATOS EN FORMATO TABLA:\n")
        f.write("-" * 100 + "\n")
        
        # Crear encabezados con porcentajes de datos disponibles
        encabezados = []
        porcentajes_encabezados = []
        
        for col in datos_reporte.columns:
            if col != 'municipio':
                datos_no_nulos = datos_reporte[col].notna().sum()
                porcentaje = (datos_no_nulos / total_registros * 100) if total_registros > 0 else 0
                encabezados.append(col)
                porcentajes_encabezados.append(f"({porcentaje:.1f}%)")
            else:
                encabezados.append(col)
                porcentajes_encabezados.append("(100%)")
        
        # Escribir encabezados principales
        encabezados_str = ""
        for i, encabezado in enumerate(encabezados):
            if i == 0:
                encabezados_str += f"{encabezado:<20}"
            else:
                encabezados_str += f"{encabezado:>15}"
        f.write(encabezados_str + "\n")
        
        # Escribir línea de porcentajes
        porcentajes_str = ""
        for i, porcentaje in enumerate(porcentajes_encabezados):
            if i == 0:
                porcentajes_str += f"{porcentaje:<20}"
            else:
                porcentajes_str += f"{porcentaje:>15}"
        f.write(porcentajes_str + "\n")
        
        # Línea separadora
        f.write("-" * 100 + "\n")
        
        # Escribir datos fila por fila con formato personalizado
        for index, row in datos_reporte.iterrows():
            fila_str = ""
            for i, col in enumerate(datos_reporte.columns):
                valor = row[col]
                if pd.isna(valor):
                    valor_str = "N/A"
                elif col == 'fecha':
                    valor_str = str(valor)
                elif col == 'municipio':
                    valor_str = str(valor)
                else:
                    # Para valores numéricos, formatear con 2 decimales
                    try:
                        valor_str = f"{float(valor):.2f}"
                    except:
                        valor_str = str(valor)
                
                if i == 0:
                    fila_str += f"{valor_str:<20}"
                else:
                    fila_str += f"{valor_str:>15}"
            
            f.write(fila_str + "\n")
        
        f.write("\n\n" + "=" * 100 + "\n")
        f.write("ESTADÍSTICAS RESUMEN:\n")
        f.write("=" * 100 + "\n")
        
        # Estadísticas básicas para columnas numéricas
        for col in datos_reporte.select_dtypes(include=[np.number]).columns:
            if col in datos_reporte.columns:
                f.write(f"\n{col.upper()}:\n")
                f.write(f"  • Promedio: {datos_reporte[col].mean():.2f}\n")
                f.write(f"  • Mínimo: {datos_reporte[col].min():.2f}\n")
                f.write(f"  • Máximo: {datos_reporte[col].max():.2f}\n")
                f.write(f"  • Valores nulos: {datos_reporte[col].isnull().sum()}\n")
        
        # Rango de fechas
        if 'fecha' in datos_reporte.columns:
            f.write(f"\nRANGO TEMPORAL:\n")
            f.write(f"  • Fecha inicial: {datos_reporte['fecha'].min()}\n")
            f.write(f"  • Fecha final: {datos_reporte['fecha'].max()}\n")
            f.write(f"  • Años de datos: {datos_reporte['fecha'].nunique()} fechas únicas\n")
    
    print(f"\nARCHIVO EXPORTADO:")
    print(f"Datos de A Gudiña guardados en: {archivo_txt}")
    print(f"Registros exportados: {len(datos_reporte):,}")
    print(f"Columnas incluidas: {', '.join(columnas_disponibles)}")
    
else:
    print("\nNo se pueden exportar datos - No se encontró A Gudiña")

Buscando municipio: "a gudiña"
Coincidencias exactas: 8401
Coincidencias parciales: 8401
Municipios que contienen "gudiña": ['a gudiña']

Primeros 5 registros de coincidencias exactas:
        municipio       fecha  temp_media  precipitacion
1816241  a gudiña  2000-01-01         1.2            0.0
1816242  a gudiña  2000-01-02         0.3            0.0
1816243  a gudiña  2000-01-03        -1.6            0.0
1816244  a gudiña  2000-01-04        -1.7            0.0
1816245  a gudiña  2000-01-05        -0.2            0.0

ARCHIVO EXPORTADO:
Datos de A Gudiña guardados en: C:\00 - Proyecto Incendios Galicia - END\data\reportes\datos_a_gudiña_clima_20250810_210601.txt
Registros exportados: 8,401
Columnas incluidas: municipio, fecha, temp_media, temp_min, temp_max, precipitacion
